In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# JupyterLab Workflow — Kernel sạch và bài nộp tái lập

Reference thực hành. Dự đoán trước mỗi experiment, rồi ghi Result → Observation → Why.

## Notebook, kernel và disk

Làm các thao tác UI trong README. Code bên dưới kiểm tra lưu/reload; `del` minh họa mất state, không thay việc bạn tự thử Restart Kernel and Run All Cells.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X = rng.normal(size=(120 if FAST else 240, 2))
y = (X[:, 0] - X[:, 1] > 0).astype(int)
X_test = rng.normal(size=(20, 2))
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])
tr, va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42)).fit(X[tr], y[tr])
valid_probability = model.predict_proba(X[va])[:, 1]
print('validation macro_f1:', f1_score(y[va], valid_probability >= 0.5, average='macro'))


## Lưu contract và state đủ dùng

Lab logistic binary lưu mean, scale, coefficient, intercept và class order. Đây là định dạng tối giản riêng cho model này, không phải serializer dùng cho mọi sklearn model.

In [ ]:
scaler = model.named_steps['standardscaler']
classifier = model.named_steps['logisticregression']
assert classifier.classes_.tolist() == [0, 1]
expected = model.predict_proba(X_test)[:, 1]
np.savez(OUT / 'workflow_state.npz', mean=scaler.mean_, scale=scaler.scale_,
         weight=classifier.coef_[0], bias=classifier.intercept_[0], classes=classifier.classes_)
np.savez(OUT / 'workflow_inputs.npz', X_test=X_test, expected=expected, ids=test_ids)
contract = {'schema_version': 1, 'features': ['x1', 'x2'],
            'prediction': 'p_class_1 >= 0.5', 'rows': len(test_ids)}
(OUT / 'workflow_contract.json').write_text(json.dumps(contract, indent=2), encoding='utf-8')
# WHY: prove that inference does not read the fitted estimator left in RAM.
del model, scaler, classifier, X_test, expected, test_ids


## Reload → Predict

Dự đoán: state trong disk còn nguyên. Chạy cell này sau cell lưu; nếu dùng kernel mới, chạy Setup trước rồi cell này với artifacts đã có.

In [ ]:
contract = json.loads((OUT / 'workflow_contract.json').read_text(encoding='utf-8'))
assert contract['schema_version'] == 1
with np.load(OUT / 'workflow_state.npz', allow_pickle=False) as artifact:
    mean, scale = artifact['mean'], artifact['scale']
    weight, bias, classes = artifact['weight'], artifact['bias'], artifact['classes']
with np.load(OUT / 'workflow_inputs.npz', allow_pickle=False) as data:
    X_test, expected, test_ids = data['X_test'], data['expected'], data['ids']
assert len(X_test) == contract['rows']
assert classes.tolist() == [0, 1]
assert mean.shape == scale.shape == weight.shape == (2,)
assert (scale > 0).all()
logits = ((X_test - mean) / scale) @ weight + bias
replayed = np.exp(-np.logaddexp(0, -logits))
assert np.allclose(replayed, expected, rtol=1e-7, atol=1e-9)
test_pred = (replayed >= 0.5).astype(int)
print('replay matches:', np.allclose(replayed, expected))


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_workflow.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_workflow.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Hypothesis → Result → Observation → Why

Thực hiện UI Restart & Run All hai lần. Output phải giữ cùng schema và predictions, nhưng thời gian có thể khác. Tự tạo cell tham chiếu biến chưa định nghĩa để nhận ra NameError, sau đó sửa thứ tự và chạy lại. Không để fault cell chưa được xử lý trong bản nộp. Xem Running panel và terminal theo checklist README; ghi rõ thao tác nào bạn thực sự đã làm.